In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType
from delta.tables import DeltaTable

CATALOG = "workspace"
SCHEMA = "oss_ecosystem"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_github_events"
SILVER_PR_TABLE = f"{CATALOG}.{SCHEMA}.silver_pull_request_events"

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.oss_ecosystem.silver_pull_request_events (
    event_id STRING,
    event_ts TIMESTAMP,
    actor_id BIGINT,
    repo_id BIGINT,
    action STRING,
    pr_number BIGINT,
    pr_id BIGINT,
    source_file STRING,
    source_hour TIMESTAMP,
    ingested_at TIMESTAMP
)
USING DELTA
""")

In [0]:
pr_payload_schema = StructType([
    StructField("action", StringType(), True),
    StructField("number", LongType(), True),
    StructField(
        "pull_request",
        StructType([
            StructField("id", LongType(), True),
            StructField("number", LongType(), True),
            StructField("url", StringType(), True)
        ]),
        True
    )
])

In [0]:
existing_pr_ids = (
    spark.table(SILVER_PR_TABLE)
    .select("event_id")
)

new_pr_bronze_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("event_type") == "PullRequestEvent")
    .join(
        existing_pr_ids,
        on="event_id",
        how="left_anti"
    )
)

print(
    "New PullRequestEvents to transform:",
    new_pr_bronze_df.count()
)

In [0]:
silver_pr_df = (
    new_pr_bronze_df
    .withColumn(
        "payload_parsed",
        F.from_json(
            F.col("payload_json"),
            pr_payload_schema
        )
    )
    .select(
        "event_id",

        F.to_timestamp("created_at")
            .alias("event_ts"),

        F.get_json_object("actor_json", "$.id")
            .cast("long")
            .alias("actor_id"),

        F.get_json_object("repo_json", "$.id")
            .cast("long")
            .alias("repo_id"),

        F.col("payload_parsed.action")
            .alias("action"),

        F.col("payload_parsed.number")
            .alias("pr_number"),

        F.col("payload_parsed.pull_request.id")
            .alias("pr_id"),

        "source_file",
        "source_hour",
        "ingested_at"
    )
)

In [0]:
pr_bronze_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("event_type") == "PullRequestEvent")
)

print("PullRequestEvent rows in Bronze:", pr_bronze_df.count())

In [0]:
print("Silver PR rows:", silver_pr_df.count())

silver_pr_df.select(
    F.sum(F.col("event_id").isNull().cast("int"))
        .alias("null_event_id"),

    F.sum(F.col("actor_id").isNull().cast("int"))
        .alias("null_actor_id"),

    F.sum(F.col("repo_id").isNull().cast("int"))
        .alias("null_repo_id"),

    F.sum(F.col("action").isNull().cast("int"))
        .alias("null_action"),

    F.sum(F.col("pr_number").isNull().cast("int"))
        .alias("null_pr_number"),

    F.sum(F.col("pr_id").isNull().cast("int"))
        .alias("null_pr_id")
).show()

In [0]:
duplicate_count = (
    silver_pr_df
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate PullRequestEvent IDs:", duplicate_count)

In [0]:
if not spark.catalog.tableExists(SILVER_PR_TABLE):
    (
        silver_pr_df.limit(0)
        .write
        .format("delta")
        .saveAsTable(SILVER_PR_TABLE)
    )

silver_pr_delta = DeltaTable.forName(
    spark,
    SILVER_PR_TABLE
)

(
    silver_pr_delta.alias("t")
    .merge(
        silver_pr_df.alias("s"),
        "t.event_id = s.event_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Silver PR MERGE completed.")

In [0]:
print(
    "Silver PR table rows:",
    spark.table(SILVER_PR_TABLE).count()
)

print(
    "Duplicate PullRequestEvent IDs:",
    spark.table(SILVER_PR_TABLE)
        .groupBy("event_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)